# 📊 Dashboard de Ventas — Multi-Sucursal

Herramienta interactiva para analizar ventas de tiendas de construcción.
Funciona con **cualquier número de sucursales**: agrega filas nuevas al CSV
(con una columna `Sede` distinta) y el dashboard las reconoce automáticamente.

**Cómo usar:** ejecuta todas las celdas (`Cell > Run All` o ▶▶). Al final aparecen
los controles — cambia la sucursal, categoría o periodo y las gráficas y KPI se actualizan solas.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import interactive_output, VBox, HBox, Box, HTML as WHTML
from IPython.display import display, clear_output

pd.options.display.float_format = '{:,.2f}'.format

## 1. Cargar y preparar datos

Cambia `CSV_PATH` por la ruta de tu archivo cuando quieras usar datos actualizados.

In [2]:
CSV_PATH = "ventas.csv"

df = pd.read_csv(CSV_PATH)

# Limpieza y columnas calculadas
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df['Mes'] = df['Date'].dt.to_period('M').astype(str)
df['Anio'] = df['Date'].dt.year
df['Margen'] = df['Net Unit Price'] - df['Unit Cost']
df['Margen_pct'] = (df['Margen'] / df['Net Unit Price']) * 100
df['Descuento_pct'] = ((df['Price List'] - df['Net Unit Price']) / df['Price List']) * 100

print(f"{len(df):,} líneas | {df['Invoice No'].nunique():,} facturas | "
      f"{df['Sede'].nunique()} sucursales: {list(df['Sede'].unique())}")
print(f"Rango de fechas: {df['Date'].min().date()} a {df['Date'].max().date()}")
df.head()

1,009 líneas | 400 facturas | 2 sucursales: ['DEL', 'PAL']
Rango de fechas: 2025-01-01 a 2026-06-27


,Sede,Invoice No,Date,Line,Product Code,Invoice Quantity,Unit Cost,Net Unit Price,Price List,Price Limit,Price Std,Category Name,Total,Mes,Anio,Margen,Margen_pct,Descuento_pct
0,DEL,DEL1000,2025-05-12,10,PLY-680,2,11.45,21.20,23.32,20.14,21.20,Plywood,42.40,2025-05,2025,9.75,45.99,9.09
1,DEL,DEL1001,2025-08-17,10,CEM-425,1,23.54,34.87,38.36,33.13,34.87,Cemento,34.87,2025-08,2025,11.33,32.49,9.10
2,DEL,DEL1001,2025-08-17,20,PLY-994,1,10.94,19.58,21.54,18.60,19.58,Plywood,19.58,2025-08,2025,8.64,44.13,9.10
3,DEL,DEL1001,2025-08-17,30,TUB-134,2,33.42,58.77,64.65,55.83,58.77,Tuberías,117.54,2025-08,2025,25.35,43.13,9.10
4,DEL,DEL1002,2026-06-23,10,PIN-721,3,22.66,32.27,35.50,30.66,32.27,Pinturas,96.81,2026-06,2026,9.61,29.78,9.10


## 2. Funciones compartidas (KPI, filtros y gráficas)

Las gráficas se construyen como **`FigureWidget`** — un solo objeto por gráfica que se
**actualiza en el mismo lugar** cada vez que cambian los filtros, en vez de crear y mostrar
una figura nueva cada vez. Esto es justo lo que evita que las gráficas se dupliquen en
VS Code/Jupyter (el problema de `fig.show()` repetido dentro de un `Output`).

In [3]:
sede_options = ['Todas'] + sorted(df['Sede'].unique().tolist())
cat_options = ['Todas'] + sorted(df['Category Name'].unique().tolist())
meses_ordenados = sorted(df['Mes'].unique())
anios_disponibles = sorted(df['Anio'].dropna().unique().astype(int).tolist())

presets = ['Todo el periodo']
presets += [f'Año {a} completo' for a in anios_disponibles]
presets += ['Últimos 3 meses', 'Últimos 6 meses', 'Últimos 12 meses', 'Personalizado']

COLORES_SEDE = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

def filtrar(sede, categoria, desde, hasta):
    d = df.copy()
    if sede != 'Todas':
        d = d[d['Sede'] == sede]
    if categoria != 'Todas':
        d = d[d['Category Name'] == categoria]
    d = d[(d['Mes'] >= desde) & (d['Mes'] <= hasta)]
    return d

def calcular_kpis(d):
    """Devuelve un dict con los KPI de un DataFrame ya filtrado."""
    ventas = d['Total'].sum()
    n_facturas = d['Invoice No'].nunique()
    ticket_prom = d.groupby('Invoice No')['Total'].sum().mean() if n_facturas else 0
    margen_pct = d['Margen_pct'].mean() if len(d) else 0
    descuento = d['Descuento_pct'].mean() if len(d) else 0
    unidades = d['Invoice Quantity'].sum()
    return {
        'Ventas': ventas, 'Facturas': n_facturas, 'Ticket prom.': ticket_prom,
        'Margen %': margen_pct, 'Descuento %': descuento, 'Unidades': unidades
    }

def html_kpi_por_sede(d, sedes_en_filtro):
    filas = []
    for s in sedes_en_filtro:
        k = calcular_kpis(d[d['Sede'] == s])
        k['Sede'] = s
        filas.append(k)
    total_k = calcular_kpis(d)
    total_k['Sede'] = 'TOTAL'
    filas.append(total_k)
    tabla = pd.DataFrame(filas)[['Sede', 'Ventas', 'Facturas', 'Ticket prom.', 'Margen %', 'Descuento %', 'Unidades']]

    def formatear(row):
        es_total = row['Sede'] == 'TOTAL'
        estilo = "font-weight:bold; background:#eaeaea;" if es_total else ""
        return (f"<tr style='{estilo}'>"
                f"<td style='padding:6px 12px;'>{row['Sede']}</td>"
                f"<td style='padding:6px 12px;'>${row['Ventas']:,.2f}</td>"
                f"<td style='padding:6px 12px;'>{row['Facturas']:,}</td>"
                f"<td style='padding:6px 12px;'>${row['Ticket prom.']:,.2f}</td>"
                f"<td style='padding:6px 12px;'>{row['Margen %']:,.1f}%</td>"
                f"<td style='padding:6px 12px;'>{row['Descuento %']:,.1f}%</td>"
                f"<td style='padding:6px 12px;'>{row['Unidades']:,}</td>"
                f"</tr>")

    filas_html = "".join(tabla.apply(formatear, axis=1))
    return f'''
    <table style="border-collapse:collapse; margin-bottom:16px; font-size:14px;">
      <thead>
        <tr style="background:#f4f4f4; text-align:left;">
          <th style="padding:6px 12px;">Sucursal</th>
          <th style="padding:6px 12px;">Ventas</th>
          <th style="padding:6px 12px;">Facturas</th>
          <th style="padding:6px 12px;">Ticket prom.</th>
          <th style="padding:6px 12px;">Margen %</th>
          <th style="padding:6px 12px;">Descuento %</th>
          <th style="padding:6px 12px;">Unidades</th>
        </tr>
      </thead>
      <tbody>{filas_html}</tbody>
    </table>
    '''

def html_kpi_tarjetas(k):
    return f'''
    <div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:16px;">
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">VENTAS TOTALES</div>
        <div style="font-size:22px; font-weight:bold;">${k['Ventas']:,.2f}</div>
      </div>
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">FACTURAS</div>
        <div style="font-size:22px; font-weight:bold;">{k['Facturas']:,}</div>
      </div>
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">TICKET PROMEDIO</div>
        <div style="font-size:22px; font-weight:bold;">${k['Ticket prom.']:,.2f}</div>
      </div>
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">MARGEN % PROMEDIO</div>
        <div style="font-size:22px; font-weight:bold;">{k['Margen %']:,.1f}%</div>
      </div>
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">DESCUENTO PROMEDIO</div>
        <div style="font-size:22px; font-weight:bold;">{k['Descuento %']:,.1f}%</div>
      </div>
      <div style="background:#f4f4f4; padding:12px 20px; border-radius:8px;">
        <div style="font-size:12px; color:#666;">UNIDADES VENDIDAS</div>
        <div style="font-size:22px; font-weight:bold;">{k['Unidades']:,}</div>
      </div>
    </div>
    '''

# ---- Constructores y actualizadores de FigureWidget (una sola instancia, se actualiza in-place) ----

def fig_ventas_mes_nueva():
    return go.FigureWidget(layout=dict(title='Ventas por mes', height=350, margin=dict(t=40, b=20),
                                        xaxis_title='Mes', yaxis_title='Ventas ($)'))

def fig_ventas_mes_actualizar(fig, d):
    ventas_mes = d.groupby(['Mes', 'Sede'])['Total'].sum().reset_index()
    with fig.batch_update():
        fig.data = []
        for i, sede in enumerate(sorted(ventas_mes['Sede'].unique())):
            sub = ventas_mes[ventas_mes['Sede'] == sede].sort_values('Mes')
            fig.add_scatter(x=sub['Mes'], y=sub['Total'], mode='lines+markers', name=sede,
                             line=dict(color=COLORES_SEDE[i % len(COLORES_SEDE)]))

def fig_categoria_nueva():
    return go.FigureWidget(layout=dict(
        title='Ventas por categoría (barras) vs. Margen % promedio (línea)',
        height=380, margin=dict(t=40, b=20),
        yaxis=dict(title='Ventas ($)'), yaxis2=dict(title='Margen %', overlaying='y', side='right')
    ))

def fig_categoria_actualizar(fig, d):
    cat_summary = d.groupby('Category Name').agg(
        Ventas=('Total', 'sum'), Margen_pct=('Margen_pct', 'mean')
    ).reset_index().sort_values('Ventas', ascending=False)
    with fig.batch_update():
        fig.data = []
        fig.add_bar(x=cat_summary['Category Name'], y=cat_summary['Ventas'], name='Ventas ($)')
        fig.add_scatter(x=cat_summary['Category Name'], y=cat_summary['Margen_pct'], name='Margen %',
                         yaxis='y2', mode='lines+markers', line=dict(color='crimson'))

def fig_heatmap_nueva():
    return go.FigureWidget(layout=dict(title='Mapa de calor — Margen % por Categoría y Sucursal',
                                        margin=dict(t=40, b=20), height=360))

def fig_heatmap_actualizar(fig, d):
    pivot = d.pivot_table(index='Category Name', columns='Sede', values='Margen_pct', aggfunc='mean')
    pivot = pivot.loc[pivot.mean(axis=1).sort_values(ascending=False).index]
    with fig.batch_update():
        fig.data = []
        fig.add_heatmap(z=pivot.values, x=list(pivot.columns), y=list(pivot.index),
                         colorscale='RdYlGn', zmin=0, zmax=max(60, pivot.values.max()),
                         text=[[f'{v:.1f}' for v in row] for row in pivot.values],
                         texttemplate='%{text}', colorbar=dict(title='Margen %'))
        fig.layout.height = max(320, 40 * len(pivot) + 100)

def fig_comparativo_nueva():
    return go.FigureWidget(layout=dict(title='Comparativo de ventas por sucursal', height=320,
                                        margin=dict(t=40, b=20), showlegend=False))

def fig_comparativo_actualizar(fig, d):
    sede_summary = d.groupby('Sede').agg(Ventas=('Total', 'sum')).reset_index()
    with fig.batch_update():
        fig.data = []
        colores = [COLORES_SEDE[i % len(COLORES_SEDE)] for i in range(len(sede_summary))]
        fig.add_bar(x=sede_summary['Sede'], y=sede_summary['Ventas'], marker=dict(color=colores),
                    text=sede_summary['Ventas'].map(lambda v: f'${v:,.0f}'), textposition='outside')

## 3. Controles interactivos

- **Sucursal / Categoría:** se llenan solos a partir de los datos.
- **Periodo:** puedes usar el selector rápido (todo el histórico, un año completo,
  últimos N meses) o elegir un mes exacto de inicio y fin en "Personalizado".

In [ ]:
sede_w = widgets.Dropdown(options=sede_options, value='Todas', description='Sucursal:')
cat_w = widgets.Dropdown(options=cat_options, value='Todas', description='Categoría:')
preset_w = widgets.Dropdown(options=presets, value='Todo el periodo', description='Periodo rápido:',
                             style={'description_width': 'initial'})
desde_w = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[0], description='Desde:')
hasta_w = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[-1], description='Hasta:')
custom_box = HBox([desde_w, hasta_w])
custom_box.layout.display = 'none'

def aplicar_preset(change=None):
    val = preset_w.value
    if val == 'Todo el periodo':
        desde_w.value, hasta_w.value = meses_ordenados[0], meses_ordenados[-1]
        custom_box.layout.display = 'none'
    elif val == 'Personalizado':
        custom_box.layout.display = 'flex'
    elif val.startswith('Año '): # pyright: ignore[reportOptionalMemberAccess]
        anio = int(val.split()[1]) # type: ignore
        meses_anio = [m for m in meses_ordenados if m.startswith(str(anio))]
        desde_w.value, hasta_w.value = meses_anio[0], meses_anio[-1]
        custom_box.layout.display = 'none'
    else:
        n = int(val.split()[1]) # pyright: ignore[reportOptionalMemberAccess]
        desde_w.value, hasta_w.value = meses_ordenados[-min(n, len(meses_ordenados))], meses_ordenados[-1]
        custom_box.layout.display = 'none'

preset_w.observe(aplicar_preset, names='value')

controles = HBox([sede_w, cat_w])
display(controles, preset_w, custom_box)

Dropdown(description='Periodo rápido:', options=('Todo el periodo', 'Año 2025 completo', 'Año 2026 completo', …

## 4. KPI y gráficas (se actualizan con los controles de arriba)

Si seleccionas **'Todas'** las sucursales, los KPI se muestran desglosados por sede y con una fila de **Total**.

In [5]:
out_kpi = widgets.Output()

fig1 = fig_ventas_mes_nueva()
fig2 = fig_categoria_nueva()
fig_heat = fig_heatmap_nueva()
fig3 = fig_comparativo_nueva()
heat_box = Box([fig_heat])
comp_box = Box([fig3])

def render(sede, categoria, preset, desde, hasta):
    d = filtrar(sede, categoria, desde, hasta)

    with out_kpi:
        clear_output(wait=True)
        if d.empty:
            print("No hay datos para este filtro.")
        else:
            periodo_label = desde if desde == hasta else f"{desde} a {hasta}"
            display(WHTML(f"<div style='color:#666; font-size:13px; margin-bottom:8px;'>Periodo: <b>{periodo_label}</b></div>"))
            sedes_en_filtro = sorted(d['Sede'].unique())
            if sede == 'Todas' and len(sedes_en_filtro) > 1:
                display(WHTML(html_kpi_por_sede(d, sedes_en_filtro)))
            else:
                display(WHTML(html_kpi_tarjetas(calcular_kpis(d))))

    if d.empty:
        return

    sedes_en_filtro = sorted(d['Sede'].unique())
    fig_ventas_mes_actualizar(fig1, d)
    fig_categoria_actualizar(fig2, d)

    if len(sedes_en_filtro) > 1:
        fig_heatmap_actualizar(fig_heat, d)
        fig_comparativo_actualizar(fig3, d)
        heat_box.layout.display = ''
        comp_box.layout.display = ''
    else:
        heat_box.layout.display = 'none'
        comp_box.layout.display = 'none'

interactive_output(render, {'sede': sede_w, 'categoria': cat_w, 'preset': preset_w,
                             'desde': desde_w, 'hasta': hasta_w})
display(VBox([out_kpi, fig1, fig2, heat_box, comp_box]))

    'data': [{'line': {'color': '#1f77b4'},
              'mode': 'lin…

## 5. Cómo escalar esto a más sucursales

- **Agregar una tienda nueva:** solo pega sus filas en el CSV con su propio código de `Sede` (por ejemplo `GYE`). No hay que tocar el código — los dropdowns y la tabla de KPI por sede se generan a partir de los datos.
- **Automatizar la carga:** si cada sucursal exporta su propio archivo, puedes concatenarlos con `pd.concat([pd.read_csv(f) for f in lista_de_archivos])` antes de la celda de limpieza.
- **Compartirlo como app (sin mostrar código):** instala `voila` (`pip install voila`) y corre `voila este_notebook.ipynb` — abre una página web con solo los controles y las gráficas, ideal para que lo use alguien que no usa Jupyter.
- **Actualización periódica:** si el CSV se sobrescribe cada semana/mes con datos nuevos, basta con volver a correr `Run All` para refrescar todo el dashboard.

---

## 6. 📊 Dashboard unificado

Todo lo anterior (secciones 3 y 4) explica el paso a paso. Esta sección junta
los mismos controles y gráficas en **una sola vista**, con un poco más de
diseño, pensada para uso diario — reutiliza las funciones de la Sección 2
(`filtrar`, `calcular_kpis`, los constructores de `FigureWidget`, etc.) con
sus propios controles y sus propias figuras, independientes de la Sección 4.

In [ ]:
# --- Controles propios de esta vista (independientes de la Sección 3) ---
sede_dash = widgets.Dropdown(options=sede_options, value='Todas', description='Sucursal:',
                              layout=widgets.Layout(width='220px'))
cat_dash = widgets.Dropdown(options=cat_options, value='Todas', description='Categoría:',
                             layout=widgets.Layout(width='220px'))
preset_dash = widgets.Dropdown(options=presets, value='Todo el periodo', description='Periodo:',
                                style={'description_width': 'initial'}, layout=widgets.Layout(width='260px'))
desde_dash = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[0], description='Desde:',
                               layout=widgets.Layout(width='160px'))
hasta_dash = widgets.Dropdown(options=meses_ordenados, value=meses_ordenados[-1], description='Hasta:',
                               layout=widgets.Layout(width='160px'))
custom_box_dash = HBox([desde_dash, hasta_dash])
custom_box_dash.layout.display = 'none'

def aplicar_preset_dash(change=None):
    val = preset_dash.value
    if val == 'Todo el periodo':
        desde_dash.value, hasta_dash.value = meses_ordenados[0], meses_ordenados[-1]
        custom_box_dash.layout.display = 'none'
    elif val == 'Personalizado':
        custom_box_dash.layout.display = 'flex'
    elif val.startswith('Año '): # pyright: ignore[reportOptionalMemberAccess]
        anio = int(val.split()[1]) # pyright: ignore[reportOptionalMemberAccess]
        meses_anio = [m for m in meses_ordenados if m.startswith(str(anio))]
        desde_dash.value, hasta_dash.value = meses_anio[0], meses_anio[-1]
        custom_box_dash.layout.display = 'none'
    else:
        n = int(val.split()[1]) # pyright: ignore[reportOptionalMemberAccess]
        desde_dash.value, hasta_dash.value = meses_ordenados[-min(n, len(meses_ordenados))], meses_ordenados[-1]
        custom_box_dash.layout.display = 'none'

preset_dash.observe(aplicar_preset_dash, names='value')

header_dash = WHTML('''
<div style="background:linear-gradient(90deg,#1f3b57,#2c6e91); color:white;
            padding:18px 24px; border-radius:12px 12px 0 0;">
  <div style="font-size:20px; font-weight:600;">📊 Panel de Ventas</div>
  <div style="opacity:0.85; font-size:13px; margin-top:2px;">
    Sucursales: {n_sedes} · Vista unificada, filtra por sucursal, categoría y periodo
  </div>
</div>
'''.format(n_sedes=df['Sede'].nunique()))

filtros_dash = Box(
    [HBox([sede_dash, cat_dash, preset_dash]), custom_box_dash],
    layout=widgets.Layout(display='flex', flex_flow='column', gap='8px',
                          border='1px solid #dfe3e6', padding='16px 20px')
)

out_kpi_dash = widgets.Output()
fig1_dash = fig_ventas_mes_nueva()
fig2_dash = fig_categoria_nueva()
fig_heat_dash = fig_heatmap_nueva()
fig3_dash = fig_comparativo_nueva()
heat_box_dash = Box([fig_heat_dash])
comp_box_dash = Box([fig3_dash])

In [7]:
def render_dash(sede, categoria, preset, desde, hasta):
    d = filtrar(sede, categoria, desde, hasta)

    with out_kpi_dash:
        clear_output(wait=True)
        if d.empty:
            print("No hay datos para este filtro.")
        else:
            periodo_label = desde if desde == hasta else f"{desde} a {hasta}"
            display(WHTML(f"<div style='color:#666; font-size:13px; margin-bottom:8px;'>Periodo: <b>{periodo_label}</b></div>"))
            sedes_en_filtro = sorted(d['Sede'].unique())
            if sede == 'Todas' and len(sedes_en_filtro) > 1:
                display(WHTML(html_kpi_por_sede(d, sedes_en_filtro)))
            else:
                display(WHTML(html_kpi_tarjetas(calcular_kpis(d))))

    if d.empty:
        return

    sedes_en_filtro = sorted(d['Sede'].unique())
    fig_ventas_mes_actualizar(fig1_dash, d)
    fig_categoria_actualizar(fig2_dash, d)

    if len(sedes_en_filtro) > 1:
        fig_heatmap_actualizar(fig_heat_dash, d)
        fig_comparativo_actualizar(fig3_dash, d)
        heat_box_dash.layout.display = ''
        comp_box_dash.layout.display = ''
    else:
        heat_box_dash.layout.display = 'none'
        comp_box_dash.layout.display = 'none'

interactive_output(render_dash, {
    'sede': sede_dash, 'categoria': cat_dash, 'preset': preset_dash,
    'desde': desde_dash, 'hasta': hasta_dash
})

display(VBox([header_dash, filtros_dash, out_kpi_dash, fig1_dash, fig2_dash, heat_box_dash, comp_box_dash]))